# MLC prototype (V0)
## Table of contents
1. [Overview](#overview)
2. [Product description](#product_description)
    - [Main variables](#main_variables)
    - [Metadata](#metadata)
3. [Input data](#input_data)
    - [SAR data](#sar_data)
    - [Auxiliary data](#auxiliary_data)
4. [Dask cluster configuration](#dask_cluster)
5. [Algorithm description](#algorithm_description)
    - [Debursting](#debursting)
    - [Timing Corrections](#timing_corrections)
    - [Thermal Noise Removal](#thermal_noise_removal)
    - [DEM Resampling](#dem_resampling)
    - [Geo2Radar Coordinates Conversion](#geo2radar_coordinates_conversion)
    - [Radiometric Terrain Correction (RTC)](#radiometric_terrain_correction)
    - [Geocoding](#geocoding)
    - [Mosaicking](#mosaicking)
6. [Data visualization](#visual)
7. [Applications](#applications)
8. [References](#references)

## Overview

The MLC product is a multilooked ARD product suited to large-scale interferometric applications. It is inspired by intermediate products which are normally present in PS/DS and SBAS chains before phase unwrapping. For the ROSE-L missions the proposed resolution of this product is 50 m. It is intended to simplify the access to interferometric measurements which are usually quite unaccessible: the product is intended to have reduced noise as it is already multilooked, it is generated with a mechanism to minimize long-term phase drifts related to vegetation effects, and it has a much smaller size compared to the original CSLC stack. The product is first generated in radar coordinates on a sub-sampling of the original SLC grid. Because of this, this version is called CMLC (Co-registered Multi-looked Complex). Its final version after geocoding is called GMLC.

There are different approaches in the literature to generate such product. Here we describe an algorithm which exhibits the following characteristics:
- it is very lightweight in terms of data I/O and computation
- it enables continuous product update as new images are acquired and the CLSC stack grows
- it incorporates a long-term drift correction


## Product Description
The multi-looked phase consists of one layer for each acquisition date of the input CSLC products. We generate it originally in radar geometry and project it to the ground at the very end. Receiving the product in ground coordinates is important for many users. (Mosaicking should also be considered). The time series of MLC phases can be used to quickly obtain interferometric maps between tow dates or relative deformation time series between two or more points.

Some of the characteristics of this product are shared with the INT and GINT products. All model-driven and data driven corrections (atmosphere, solid earth tides, etc.) can be applied to this product, if they are not already included in the input CSLCs.

The target posting is 50 m, in line with the ROSE-L MRD. Advanced realizations could consider the use of adaptive multi-looking, either with older methods for homogeneous pixel selection or newer machine learning techniques. These are not considered at the moment.

This product includes two quality layers in the form of coherence. Quality layers can be very useful for the user for example to guide space/time phase unwrapping.

The GMLC ground grid is defined to ensure compatibility between missions and products.

### Main variables
- Wrapped multilooked phase
- Long-term coherence
- Short-term coherence

### Metadata
...

## Input data

### SAR data
...

### Auxiliary data
...

## Dask cluster configuration

## Algorithm description



![Workflow for CMLC generation.](./images/CMLC_generation.png)

### Phase estimation

The multilooked phase is going to be generated by interfering each incoming CSLC acquisition $y_n$ with the current reference $z_n$ (a “running reference”) and averaging over a small pixel neighborhood $\Omega$:
$$
   \phi_n = \angle \sum_{(r,a) \in \Omega} z_n^*(r,a) y_n(r,a).
$$

In our implementation the neighborhood is simply a rectangular window.
At the very beginning, the reference $z_n$ is going to be initialized with the first image in the CSCL stack (i.e. $z_1 = y_1$). Later it is going to be updated as described below.

### Running reference update
After the estimation of the interferometric phase $\phi_n$, we are going to use it to update the running reference $z_n$ by including the contribution of the current CLSC $y_n.$

$$
z_{n+1} = \alpha z_n + y_n e^{-j\phi_n}.
$$

The memory parameter $\alpha$ controls the bandwidth of the running reference, a small parameter corresponds to a short memory, which is optimal for fast-decorrelating scenarios. A large value of the parameter is suited to stable targets.

### Running reference calibration

### Geocoding



### Mosaicking


## Interferogram generation in QGIS

$$ \mathrm{atan2}\left(\sin(\theta), \cos(\theta)\right) = 2\:\mathrm{atan}\left(\frac{\sin(\theta)}{1+\cos(\theta)}\right)$$

## Code

In [112]:
import zarr
import numpy as np
from tqdm import tqdm

In [113]:
base_dir = "/media/data/SAR/L2/Naples/ASC/S4/"

In [114]:
# selecting window sizes (rg, az) and mini_size (number of acquisitions after which the long-term calibration kicks-in)
w_rg = 10
w_az = 20
mini_size = 5

In [115]:
# opening the CSLC dataset
ds = zarr.open(base_dir + "slcs.zarr")
cstack = ds.coreg.cstack
N = len(cstack)

In [116]:
# choosing the indices of one window (normally we are going to iterate over all window indices)
i_rg = 100
i_az = 500

In [117]:
# building indices to cut a slice for the selected window
rg_range = np.arange((i_rg-1)*w_rg+0,(i_rg)*w_rg+0)
az_range = np.arange((i_az-1)*w_az+0,(i_az)*w_az+0)
wnd_range = np.ix_(az_range, rg_range)

In [118]:
# inizializing output arrays
interf = np.zeros(N, dtype=np.complex64)
short_term_coherence = np.zeros(N)*np.nan
long_term_coherence = np.zeros(N)*np.nan

In [119]:
def make_interf(A, B):
    return np.conj(A).dot(B)

def make_coh(A, B):
    return make_interf(A,B) / np.sqrt(make_interf(A,A) * make_interf(B,B))

In [ ]:
# main algorithm (progressive interferometric phase estimation)

# initializing references (internal variables)
#current_reference = cstack[0][wnd_range].ravel()
current_reference = cstack[N-1][wnd_range].ravel()
past_reference = np.zeros_like(current_reference)

step = 0
#for n in tqdm(range(0,N)):
for n in tqdm(range(N-1,-1,-1)):
    step = step + 1
    slc = cstack[n][wnd_range].ravel()  # getting the current slc
    interf[n] = make_coh(current_reference,slc)
    short_term_coherence[n] = abs(interf[n])
    
    # update the current reference
    current_reference = current_reference*0.8 + slc*np.conj(interf[n])

    # generate (a) the past reference
    if step == mini_size :
        past_reference[:] = current_reference
        
    # long-term phase calibration of current_reference
    if step > mini_size :
        corr_phase = make_coh(past_reference,current_reference)
        if np.abs(corr_phase) > 0.1 :
            current_reference[:] = current_reference * (np.conj(corr_phase)/np.abs(corr_phase))
            long_term_coherence[n] = np.abs(make_coh(past_reference,slc))


100%|██████████| 31/31 [06:02<00:00, 11.69s/it]


In [ ]:
# collect all data in a matrix
data = np.zeros([N, w_rg*w_az], dtype=np.complex64)
for n in tdqm(range(0,N)):
    data[n,:] = cstack[n][wnd_range].flatten()

In [44]:
def phase_linking(A) :
    W = np.linalg.inv(np.abs(A))
    eigenvalues, eigenvectors = np.linalg.eigh(A * W)
    v = eigenvectors[:,0]
    v = v / np.abs(v)
    v = v * np.conj(v[0])
    return v

In [133]:
# phase linking
cov = np.matmul(data.conj(), data.T)
interf_emi = phase_linking(cov + 0.01*np.linalg.eigvalsh(cov)[N-1]*np.eye(N))

## References
